## Overview
A React Component is a reusable piece of React application. They accept arbitrary inputs (called “props”) and return React Elements describing what should appear on the screen. The code below describes how to define a component:

In [ ]:
// Component function or class name must be capitalised
function Welcome(props) {
    // Return React element
    return (
        <div>
            <h2>Welcome</h2>
            <p>to my React application</p>
        </div>
    )
}

Often, Arrow functions are employed:

In [ ]:
const Welcome = (props) => {
    return (
        <div>
            <h2>Welcome</h2>
            <p>to my React application</p>
        </div>
    )
}

React can then render the component:

In [ ]:
ReactDOM.render(<Welcome />, document.getElementById("root"));

// We don't directly call the function like below (even though technically in this case it would work):
ReactDOM.render(Welcome(), document.getElementById("root"));

// Or in React 18+
const root = ReactDOM.createRoot(document.getElementById("root"));
root.render(<Welcome />);

Internally, react would:

In [ ]:
let type = (<Welcome />).type; // type property for component is the function itself
let props = (<Welcome />).props;

let result = type(props); // call the component function

## Props
Props is how data is transfered from parent to child component. It contains all the attributes passed to the component plus all children nodes. In the example below `CountryDisplay` component passes list of countries to `CountryList` child component

In [ ]:
function CountryDisplay(props) {
    const countries = ["Japan", "Slovenia", "South Africa"];
    
    return <CountryList countries={countries} standard="ISO" />
}

The child component can then access these props:

In [ ]:
function CountryList(props) {
    const list = props.countries.map((country, index) => <li key={index}>{country}</li>)
    
    return <ul>{list}</ul>
}

// Common pattern to access only the required prop
const CountryList = ({ countries }) => { // now instead of `props.countries`, we use `countries`
    // implementation
}

React components must act like [pure functions](https://en.wikipedia.org/wiki/Pure_function) with respect to their props. This means that inside a component, props must not be changed. [Why?](https://stackoverflow.com/questions/57660115/is-the-reason-props-in-react-shouldnt-be-changed-mutated-is-because-react-wants) If a component could mutate its props, we would be changing an object that is accessible to the parent node, even after the parent node had already rendered. Consider:

In [ ]:
const Parent = (props) => {
    return (
        <div>
            {props.user.name}
            <Child user={props.user} />
        </div>
    )
}

const Child = (props) => {
    props.user.name = "Vue user";
    return <div>{props.user.name}</div>
}

React.render(<Parent user={{name: "React user"}} />, document.getElementById('container'));

When we render the Parent, it outputs "React user" to the Shadow DOM. When Child is rendered, it outputs "Vue user". Parent is now wrong. It should say "Vue user" because user.name is "Vue user", but we already output "React user", and it's too late to change it.

When a component needs to change prop it needs to request the parent to send in a new prop.

### Children Prop
React allows you to nest components within components, like:

In [ ]:
<Gallery>
  <Image />
</Gallery>

The parent component (`<Gallery>`) receives the child component (`<Image>`) in a prop called `children`:

In [ ]:
function Gallery({ children }) {
  return (
    <div className="card">
      {children}
    </div>
  );
}

export default function Page() {
  return (
    <Gallery>
      <Image
        size={100}
        src="https://example.com/img.jpg"
      />
    </Gallery>
  );
}

### Prop Validation
Often we would want to restrict type and requirement of props. We can achieve this using the `prop-types` library. Any of the following code blocks are valid way of specifying prop types:

In [ ]:
// functional component
function ReactComponent(props) {
  // implement render logic here
}
ReactComponent.propTypes = {
  // prop type definitions here
}

When props are passed to a React component, they are checked against the type definitions configured in the `propTypes` property. When an invalid value is passed for a prop, a warning is displayed in the JavaScript console.

In [ ]:
import React from 'react';
import PropTypes from 'prop-types';

const MyComponent = ({ 
    name, 
    value, 
    count, 
    onUpdate, 
    tags, 
    location, 
    identifier, 
    hideDetails, 
    banner, 
    region, 
    color, 
    company, 
    graphics, 
    calculation, 
    profile, 
    subject 
}) => {
    return (
        <div>
            {/* Your component logic and JSX go here */}
            <h1>{name}</h1>
        </div>
    );
};

MyComponent.propTypes = {
    /* Basic PropTypes */
    name: PropTypes.string,
    value: PropTypes.any,
    count: PropTypes.number,
    onUpdate: PropTypes.func,
    tags: PropTypes.array,
    location: PropTypes.object,
    identifier: PropTypes.symbol,
    hideDetails: PropTypes.bool,

    /* Renderable PropTypes */
    banner: PropTypes.element,

    /* Instance PropType */
    region: PropTypes.instanceOf(Region),

    /* Multiple types */
    color: PropTypes.oneOf(['red', 'blue', 'green', null]),
    company: PropTypes.oneOfType([
        PropTypes.object,
        PropTypes.string,
        PropTypes.instanceOf(Company)
    ]),

    /* Collection Types */
    graphics: PropTypes.arrayOf(PropTypes.string),
    calculation: PropTypes.objectOf(PropTypes.number),
    profile: PropTypes.shape({
        id: PropTypes.number,
        fullname: PropTypes.string,
        gender: PropTypes.oneOf(['M', 'F']),
        birthdate: PropTypes.instanceOf(Date),
        isAuthor: PropTypes.bool
    }),
    subject: PropTypes.exact({
        subject: PropTypes.oneOf(['Maths', 'Arts', 'Science']),
        score: PropTypes.number
    })
};

All the above props are optional. If some prop is mandatory, we can append `isRequired`, like:

In [ ]:
MyComponent.propTypes = {
    /* Optional */
    name: PropTypes.string,
    
    /* Mandatory */
    value: PropTypes.number.isRequired
}

We can also pass a custom validation function:

In [ ]:
// Validator function requires the below three parameters
const isEmail = function(props, propName, componentName) {
  const regex = /^((([^<>()[]\.,;:s@"]+(.[^<>()[]\.,;:s@"]+)*)|(".+"))@(([[0-9]{1,3}.[0-9]{1,3}.[0-9]{1,3}.[0-9]{1,3}])|(([a-zA-Z-0-9]+.)+[a-zA-Z]{2,})))?$/;
  
  if (!regex.test(props[propName])) {
    return new Error(`Invalid prop `${propName}` passed to `${componentName}`. Expected a valid email address.`);
  }
}

MyComponent.propTypes = {
  email: isEmail
}

Custom validation functions can be mixed with other prop types:

In [ ]:
MyComponent.propTypes = {
  emails: PropTypes.arrayOf(isEmail)
}

// Or
MyComponent.propTypes = {
  email: PropTypes.oneOfType([
    isEmail,
    PropTypes.shape({
      address: isEmail
    })
  ])
}

## State
State is a component's private data. Let's try with a naive way to store state:

In [ ]:
var state = {
  name: "John Doe",
  age: 35,
  active: true
}

function Person(){
  return (
    <>
      <p><b>Name:</b> {state.name} </p>
      <p><b>Age:</b> {state.age} </p>
      <p><b>Active:</b> {state.active + ""} </p>
      <button onClick = { () => {
          state.active = !state.active
          console.log(state.active)
      }}>Toggle Activation </button>
    </>
  )
}

The problem with the above approach is that even when the state is being updated, React doesn’t know that it has been updated. Therefore the component is not re-rendered.

Instead, we use `setState` method provided by React to update state. We pass partial state to the method, which is then merged with the complete state. 

In [ ]:
import { useState } from "react"

function TextInput(){
    // useState function accepts initial value as the parameter
    const [text, setText] = useState("")
    
    return (
        <>
          <input onChange={ e => setText(e.target.value) } />
          <button onClick={ () => setText("") }>Clear</button>
        </>
    )
}

The initial value will be assigned only on the initial render, subsequent renders (due to a change of state in the component or a parent component), the argument of the useState Hook will be ignored and the current value will be retrieved. 

Whenever we call the `setText` function, React persists the new value of the state and knows that the component should be re-rendered. We can pass any initial value including objects. The thing to know is that setter function of `useState` hook does a complete replacement of value. Thus, to do partial updates:

In [ ]:
const [message, setMessage] = useState({ message: '', id: 1 });

function updateMessage(event) {
    setMessage(prevState => {
        return {
            ...prevState, // copy the object
            message: event.target.value // change what you need
        }
    })
}

Sometimes the initial value may be calculated using an expensive operation, for example:

In [ ]:
function MyComponent() {
	function expensiveOperation() {
		// return some value after the expensive operation
	}

	const [state, setState] = useState(expensiveOperation())

  // rest of the operation
}

The problem with this approach is that the expensive operation function will be called whenever the `useState` function is called, even though the result of that function is used only once (to set initial value of state). React allows us to pass a function to `useState` and that function gets called only once on initial render to obtain the initial value of state.

In [ ]:
function MyComponent() {
	function expensiveOperation() {
		// return some value after the expensive operation
	}

	const [state, setState] = useState(() => expensiveOperation())

  // rest of the operation
}

If we want to do changes based on previous value:

In [ ]:
function Counter(){
  const [number, setNumber] = useState(0)

  return (
    <>
      Counter value: { value }
      <button onClick = {() => setNumber(number+1)} >Increment</button> 
      <button onClick = {() => {
        setNumber(number+1)
        setNumber(number+1)
      }}>Double</button>
    </>
  )
}

The increment functionality works as expected, however the double functionality doesn’t. The effect of the `setNumber` function is not immediate and both `setNumber` calls get the same value of number passed to them.

React may batch multiple `setState()` calls into a single update for performance. The main idea is that no matter how many `setState` calls we make inside a React event handler or synchronous lifecycle method, it will be batched into a single update. That is only one single re-render will eventually happen.

Note that this batch update functionality does not always work. In an event handler that uses an asynchronous operation of different kinds, such as `async`/`await`, `then`/`catch`, `setTimeout`, `fetch`, etc, separate state updates will not be batched. More on this [here](https://medium.com/swlh/react-state-batch-update-b1b61bd28cd2).

So, the correct way is to pass a function to the setter method, like:

In [ ]:
function Counter(){
  const [number, setNumber] = useState(0)

  return (
    <>
      Counter value: { value }
      <button onClick = {() => setNumber(previousNumber => previousNumber + 1)} >
				Increment</button> 
      
			{ /* Double functionality works now */ }	
			<button onClick = {() => {
		        setNumber(previousNumber => previousNumber + 1)
		        setNumber(previousNumber => previousNumber + 1)
		   }}>Double</button>
    </>
  )
}

### State Preservation
React keeps the state of a component around for as long as we render the same component at the same position in the tree. When a component gets removed, react destroys its state as well.

In [ ]:
function Section() {
    const [showFooter, setShowFooter] = useState(true);
    
    return (
        <>
            <Header>My Header Here</Header>
            {showFooter && <Footer>Some disclaimer text goes here.</Footer>}
            <label>
                <input
                  type="checkbox"
                  checked={showFooter}
                  onChange={e => {
                    setShowFooter(e.target.checked)
                  }}
            />Show the footer</label>
        </>
    );
}

Here if the checkbox us unchecked, the `<Footer>` component is removed along with any state it may have. But what about the below code?

In [ ]:
function Section() {
    const [bold, setBold] = useState(true);
    
    return (
        <>
            {bold ? (
                <Header><b>Bold Header Goes Here</b></Header>
            ) : (
                <Header>Normal Header Goes Here</Header>
            )}
            <label>
                <input
                  type="checkbox"
                  checked={bold}
                  onChange={e => {
                    setBold(e.target.checked)
                  }}
            />Bold?</label>
        </>
    );
}

In the above case the same component `<Header>` is displayed at the same position, therefore state is preserved across renders. But what if we wanted the two components to be treated as two distinct component? Then we need to add `key` prop.

## Context
Context provides a way to pass data through the component tree without having to pass props down manually at every level. Things such as Theme, Locale, etc can be made available in all child components without explicitly passing them as props.

We first create context using `createContext` and then provide the context:

In [ ]:
const ThemeContext = createContext();

export default function App() {
  const [theme, setTheme] = useState('light');

  const toggleTheme = () => {
    setTheme(prev => prev === 'light' ? 'dark' : 'light');
  };

  return (
    <ThemeContext.Provider value={{ theme, toggleTheme }}>
      <div style={{ background: theme === 'light' ? '#fff' : '#333', height: '100vh' }}>
        <Navbar />
      </div>
    </ThemeContext.Provider>
  );
}

Now any child component can access the context using `useContext`:

In [ ]:
function Navbar() {
  return <nav><NavButton /></nav>; // Navbar doesn't need the theme prop!
}

function NavButton() {
  const { theme, toggleTheme } = useContext(ThemeContext);

  return (
    <button onClick={toggleTheme}>
      Switch to {theme === 'light' ? 'Dark' : 'Light'} Mode
    </button>
  );
}